# 01 · Company Selection

Choose the companies whose 10-K **Item 1A (Risk Factors)** sections we will analyse.

**Design goal.** For dynamic topic modelling we want a panel that is (a) as *long* as the
data reasonably allows and (b) *dense* every year, so topic-frequency-over-time is stable.
We therefore keep only S&P 500 companies that filed a 10-K in **every year 2010–2025**, then
draw a **sector-balanced random sample** so no single industry dominates the discovered topics.

Output: `data/selected_companies.csv`.

In [1]:
# Auto-reload edited src/ modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # make `src` importable from notebooks/

import pandas as pd
from tqdm.auto import tqdm
from src import edgar

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Configuration

In [2]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2025

N_PER_SECTOR = 5        # sector-balanced sample 
SEED         = 42       

AVAIL_CSV    = DATA_DIR / "availability.csv"          # cached coverage matrix
SELECTED_CSV = DATA_DIR / "selected_companies.csv"   # final output
print(f"Study period: {START_YEAR}-{END_YEAR} (a 10-K required in every year).")

Study period: 2010-2025 (a 10-K required in every year).


## 2 · S&P 500 universe

In [3]:
companies = edgar.get_sp500()
print(f"S&P 500 constituents: {len(companies)}")
companies.head()

S&P 500 constituents: 503


,ticker,company,sector,cik
0,MMM,3M,Industrials,0000066740
1,AOS,A. O. Smith,Industrials,0000091142
2,ABT,Abbott Laboratories,Health Care,0000001800
3,ABBV,AbbVie,Health Care,0001551152
4,ACN,Accenture,Information Technology,0001467373


## 3 · 10-K coverage matrix (full history)

For each company we record which years it filed a 10-K, using `edgar.list_10k_years`
(one lightweight request per company that returns the **full** history — not just the
most-recent ~1,000 filings, which for active filers like banks does not reach back to 2010).

The result is cached to `data/availability.csv`; delete that file to force a refresh.

In [4]:
if AVAIL_CSV.exists():
    avail = pd.read_csv(AVAIL_CSV, dtype={"cik": str})
    print(f"Loaded cached coverage matrix: {avail.shape}")
else:
    records = []
    for _, c in tqdm(companies.iterrows(), total=len(companies), desc="Scanning"):
        try:
            years = set(edgar.list_10k_years(c["cik"]))
        except Exception:
            years = set()
        rec = {"ticker": c["ticker"], "company": c["company"],
               "sector": c["sector"], "cik": c["cik"]}
        for y in range(START_YEAR, END_YEAR + 1):
            rec[str(y)] = int(y in years)
        records.append(rec)
    avail = pd.DataFrame(records)
    avail.to_csv(AVAIL_CSV, index=False)
    print(f"Built and cached coverage matrix: {avail.shape}")

year_cols = [str(y) for y in range(START_YEAR, END_YEAR + 1)]
print("\n10-K filers per year:")
print(avail[year_cols].sum().to_string())

Loaded cached coverage matrix: (503, 20)

10-K filers per year:
2010    403
2011    412
2012    417
2013    426
2014    433
2015    441
2016    446
2017    449
2018    458
2019    468
2020    474
2021    481
2022    486
2023    488
2024    492
2025    498


## 4 · Filter to complete coverage

Keep companies with a 10-K in **every** year `START_YEAR..END_YEAR` (no gaps).

In [5]:
window = [str(y) for y in range(START_YEAR, END_YEAR + 1)]
eligible = avail[avail[window].sum(axis=1) == len(window)].copy()

print(f"Eligible (full {START_YEAR}-{END_YEAR} coverage): {len(eligible)} companies")
print("\nBy sector:")
print(eligible["sector"].value_counts().sort_index().to_string())

Eligible (full 2010-2025 coverage): 396 companies

By sector:
sector
Communication Services    11
Consumer Discretionary    40
Consumer Staples          32
Energy                    13
Financials                61
Health Care               47
Industrials               65
Information Technology    53
Materials                 17
Real Estate               29
Utilities                 28


## 5 · Sector-balanced random sample

In [6]:
parts = [g.sample(min(len(g), N_PER_SECTOR), random_state=SEED)
         for _, g in eligible.groupby("sector")]
selection = pd.concat(parts, ignore_index=True)

# transparency columns
hist_cols = [c for c in avail.columns if c.isdigit()]
selection["first_year"] = selection[hist_cols].apply(
    lambda r: min(int(c) for c in hist_cols if r[c] == 1), axis=1)
selection["last_year"]  = selection[hist_cols].apply(
    lambda r: max(int(c) for c in hist_cols if r[c] == 1), axis=1)
selection["n_10k"]      = selection[hist_cols].sum(axis=1).astype(int)

selected = (selection[["ticker", "company", "sector", "cik",
                       "first_year", "last_year", "n_10k"]]
            .sort_values(["sector", "ticker"]).reset_index(drop=True))
selected.to_csv(SELECTED_CSV, index=False)

print(f"Selected {len(selected)} companies -> {SELECTED_CSV.relative_to(ROOT)}")
print("\nBy sector:")
print(selected["sector"].value_counts().sort_index().to_string())
selected.head(12)

Selected 55 companies -> data/selected_companies.csv

By sector:
sector
Communication Services    5
Consumer Discretionary    5
Consumer Staples          5
Energy                    5
Financials                5
Health Care               5
Industrials               5
Information Technology    5
Materials                 5
Real Estate               5
Utilities                 5


,ticker,company,sector,cik,first_year,last_year,n_10k
0,CMCSA,Comcast,Communication Services,0001166691,2010,2025,16
1,LYV,Live Nation Entertainment,Communication Services,0001335258,2010,2025,16
2,T,AT&T,Communication Services,0000732717,2010,2025,16
3,VZ,Verizon,Communication Services,0000732712,2010,2025,16
4,WBD,Warner Bros. Discovery,Communication Services,0001437107,2010,2025,16
5,CMG,Chipotle Mexican Grill,Consumer Discretionary,0001058090,2010,2025,16
6,HAS,Hasbro,Consumer Discretionary,0000046080,2010,2025,16
7,HD,Home Depot (The),Consumer Discretionary,0000354950,2010,2025,16
8,LOW,Lowe's,Consumer Discretionary,0000060667,2010,2025,16
9,ORLY,O’Reilly Automotive,Consumer Discretionary,0000898173,2010,2025,16


## 6 · Spot-check one company

Confirm the full 10-K history is visible (accession numbers + document names) before
the extraction notebook downloads them.

In [7]:
row = selected.iloc[0]
print(f"{row['company']} ({row['ticker']}, CIK {row['cik']})")
edgar.list_10k_filings(row["cik"], START_YEAR, END_YEAR)

Comcast (CMCSA, CIK 0001166691)


,form,filing_date,accession_number,primary_document,year
0,10-K,2010-02-23,0001193125-10-037551,d10k.htm,2010
1,10-K,2011-02-25,0001193125-11-047243,d10k.htm,2011
2,10-K,2012-02-23,0001193125-12-073905,d262998d10k.htm,2012
3,10-K,2013-02-21,0001193125-13-067658,d458593d10k.htm,2013
4,10-K,2014-02-12,0001193125-14-047522,d666576d10k.htm,2014
5,10-K,2015-02-27,0001193125-15-068526,d817352d10k.htm,2015
6,10-K,2016-02-05,0001193125-16-452423,d49239d10k.htm,2016
7,10-K,2017-02-03,0001193125-17-030512,d290430d10k.htm,2017
8,10-K,2018-01-31,0001166691-18-000004,cmcsa-12312017x10k.htm,2018
9,10-K,2019-01-31,0001166691-19-000005,cmcsa-12312018x10k.htm,2019
